# Crée ton Chatbot et Pilote les Paramètres d'un LLM
### Maîtriser la Température, le Top-K et le Top-P avec Google Gemini

---

## Objectifs pédagogiques
À la fin de ce lab, vous serez capable de :
1. **Comprendre la mécanique de sélection des mots** (des probabilités brutes à l'échantillonnage final).
2. **Manipuler concrètement l'API officielle de Google Gemini** en Python.
3. **Maîtriser et configurer les paramètres de génération** (`Temperature`, `Top-K`, `Top-P`) pour orienter le comportement d'un modèle (du robot ultra-strict à l'écrivain créatif).

## Étape 1 : Récupérer sa Clé API Gratuite

Pour envoyer des requêtes à Google Gemini, nous avons besoin d'une clé d'authentification (Clé API). Google propose un accès gratuit pour tester ses modèles.

### Procédure pas à pas :
1. Rendez-vous sur **[Google AI Studio](https://aistudio.google.com/)**.
2. Connectez-vous avec votre compte Google standard (Gmail).
3. Cliquez sur le bouton bleu **"Get API key"** (Obtenir une clé API) en haut à gauche.
4. Cliquez sur **"Create API key"**.
5. Sélectionnez un projet Google Cloud (ou laissez par défaut) et copiez la clé générée (elle commence généralement par `AIzaSy...`).


In [ ]:
# =====================================================================
# ÉTAPE 2 : INSTALLATION ET CONFIGURATION DU CLIENT GEMINI
# =====================================================================

# 1. Installation de la bibliothèque officielle la plus récente de Google
!pip install -q -U google-genai

import os
from google import genai
from google.genai import types

# 2. Configuration de votre clé API
# REMPLACEZ LA VALEUR CI-DESSOUS PAR VOTRE CLÉ API RECOPIÉE DEPUIS GOOGLE AI STUDIO
API_KEY = "this is private"

# Initialisation sécurisée du client
if API_KEY == "VOTRE_CLE_ICI" or API_KEY == "":
    print("ERREUR : Vous devez remplacer 'VOTRE_CLE_ICI' par votre clé API Gemini !")
    client = None
else:
    try:
        client = genai.Client(api_key=API_KEY)
        print("Succès ! Le client Google Gemini est initialisé et prêt à l'emploi.")
    except Exception as e:
        print(f"Erreur lors de l'initialisation : {e}")
        client = None


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\vargnomad\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Succès ! Le client Google Gemini est initialisé et prêt à l'emploi.


## Rappel Théorique Éclair : Le Pipeline de Génération d'un LLM

Un LLM ne pense pas. À chaque étape, il calcule la probabilité de tous les mots possibles de son dictionnaire pour deviner le mot suivant. 

Voici comment les paramètres influencent cette décision :

```
[ LOGITS (Scores bruts de tous les mots du dictionnaire) ]
                       │
                       ▼
         [ TEMPÉRATURE (Aplatit ou étire les probabilités) ]
                       │
                       ▼
         [ TOP-K (Garde uniquement les K mots les plus probables) ]
                       │
                       ▼
         [ TOP-P (Garde le plus petit groupe dont la somme des probabilités >= P) ]
                       │
                       ▼
         [ ÉCHANTILLONNAGE FINAL (Tirage au sort parmi les survivants) ]
```

- **Température ($T$) :** Plus elle est basse (proche de 0), plus le modèle choisit de manière prévisible le mot le plus probable. Plus elle est haute (proche de 2), plus le modèle égalise les chances de tous les mots, favorisant l'originalité (et le risque d'incohérence/hallucination).
- **Top-K ($K$) :** Limite le choix aux $K$ meilleurs mots. Si $K=1$, c'est toujours le meilleur mot qui sort (mode déterministe).
- **Top-P ($P$) :** Filtre dynamique. On cumule les probabilités des mots triés par ordre décroissant et on ne garde que ceux qui entrent dans le pourcentage supérieur défini par $P$ (ex: $P=0.90$ pour garder les mots représentant 90% de la probabilité cumulée).

In [2]:
# =====================================================================
# ÉTAPE 3 : FONCTION UNIVERSELLE DU CHATBOT
# =====================================================================

def generer_reponse_ia(prompt, temperature=1.0, top_k=None, top_p=None):
    """
    Envoie une requête au modèle 'gemini-2.5-flash' avec des paramètres personnalisés.
    
    Parameters:
        prompt (str): La consigne textuelle transmise à l'IA.
        temperature (float): Degré de créativité / hasard (valeur recommandée entre 0.0 et 2.0).
        top_k (int): Nombre maximum de mots candidats conservés lors de la sélection.
        top_p (float): Seuil de probabilité cumulée pour l'échantillonnage (entre 0.0 et 1.0).
        
    Returns:
        str: Le texte généré par l'IA ou None en cas d'erreur.
    """
    if client is None:
        print("Le client API n'est pas configuré. Veuillez exécuter correctement l'étape 2.")
        return None
        
    try:
        # Définition de la configuration de génération
        config = types.GenerateContentConfig(
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            max_output_tokens=150 # Limite la réponse pour économiser nos tests
        )
        
        # Appel du modèle léger et rapide de dernière génération
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
            config=config,
        )
        return response.text
        
    except Exception as e:
        print(f"Une erreur de communication avec Gemini est survenue : {e}")
        return None

## Expérience 1 : Le Mode Déterministe (Le Robot Strict)

**Objectif :** Observer le comportement du modèle lorsque le hasard est réduit au minimum.

Ici, nous configurons :
- Une température très basse (`temperature=0.1`)
- Un `top_k=1` (seul le mot ayant la probabilité maximale absolue est éligible)
- Un `top_p=0.1` (le filtre est extrêmement étroit)

**Consigne :** Exécutez la cellule de code ci-dessous au moins **3 fois**. Comparez les résultats générés à chaque essai.

In [3]:
# Prompt créatif pour tester l'uniformité de la réponse
prompt_experience = "Raconte-moi une histoire de 3 phrases sur un chat astronaute."

print("--- APPLICATION DES PARAMÈTRES STRICTS (Temp=0.1, Top-K=1, Top-P=0.1) ---")
for i in range(1, 4):
    print(f"\n Essai {i} :")
    reponse = generer_reponse_ia(
        prompt_experience,
        temperature=0.1,
        top_k=1,
        top_p=0.1
    )
    print(reponse)

--- APPLICATION DES PARAMÈTRES STRICTS (Temp=0.1, Top-K=1, Top-P=0.1) ---

 Essai 1 :
Une erreur de communication avec Gemini est survenue : 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
None

 Essai 2 :
Une erreur de communication avec Gemini est survenue : 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
None

 Essai 3 :
Une erreur de communication avec Gemini est survenue : 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
None


##  Expérience 2 : La Température Haute (La Créativité sans Filtre)

**Objectif :** Libérer le hasard au maximum pour forcer l'originalité.

Ici, nous configurons :
- Une température très haute (`temperature=1.8`)
- Un `top_k=100` (on garde un grand éventail de mots possibles)
- Un `top_p=1.0` (on n'applique aucun filtrage dynamique par probabilité)

 **Consigne :** Exécutez cette cellule **3 fois**. Observez la diversité du vocabulaire et la structure des phrases d'un essai à l'autre.

In [4]:
print("--- APPLICATION DES PARAMÈTRES CRÉATIFS (Temp=1.8, Top-K=100, Top-P=1.0) ---")
for i in range(1, 4):
    print(f"\n Essai {i} (Créatif) :")
    reponse = generer_reponse_ia(
        prompt_experience,
        temperature=1.8,
        top_k=100,
        top_p=1.0
    )
    print(reponse)

--- APPLICATION DES PARAMÈTRES CRÉATIFS (Temp=1.8, Top-K=100, Top-P=1.0) ---

 Essai 1 (Créatif) :
Une erreur de communication avec Gemini est survenue : 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
None

 Essai 2 (Créatif) :
Une erreur de communication avec Gemini est survenue : 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
None

 Essai 3 (Créatif) :
Une erreur de communication avec Gemini est survenue : 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.'

##  Expérience 3 : Le Juste Milieu (Créatif mais Sécurisé)

**Objectif :** Garder de l'originalité tout en évitant les dérives absurdes.

Si on laisse la température à `1.4` (assez élevée) mais qu'on serre la vis avec :
- Un `top_k=20` (on supprime complètement les mots farfelus en ne gardant que les 20 meilleurs)
- Un `top_p=0.85` (on élimine la longue traîne des mots peu probables représentant les derniers 15% de chance)

On obtient une écriture fluide, vivante, diversifiée d'un essai à l'autre, mais qui respecte scrupuleusement la cohérence linguistique.

 **Consigne :** Exécutez cette cellule **2 ou 3 fois** pour valider l'équilibre trouvé.

In [5]:
print("--- APPLICATION DU JUSTE MILIEU (Temp=1.4, Top-K=20, Top-P=0.85) ---")
for i in range(1, 4):
    print(f"\n✨ Essai {i} (Équilibré) :")
    reponse = generer_reponse_ia(
        prompt=prompt_experience,
        temperature=1.4,
        top_k=20,
        top_p=0.85
    )
    print(reponse)

--- APPLICATION DU JUSTE MILIEU (Temp=1.4, Top-K=20, Top-P=0.85) ---

✨ Essai 1 (Équilibré) :
Une erreur de communication avec Gemini est survenue : 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 9.13884942s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'q

##  Mini-Quiz d'Évaluation (Questions de Synthèse)

Pour valider votre compréhension de ce laboratoire pratique, prenez le temps de répondre mentalement ou sur un bloc-notes à ces questions :

###  Question 1
Pourquoi, lors de l'Expérience 1 avec une température de `0.1`, le texte généré ne changeait pas (ou presque pas) d'un essai à l'autre ? Quelle est la mécanique sous-jacente ?

###  Question 2
Quelle est la différence fondamentale de comportement entre le **Top-K** (qui utilise un nombre fixe) et le **Top-P** (qui utilise un pourcentage) pour trier les mots ? Pourquoi le Top-P est-il souvent qualifié d'échantillonnage dynamique ?

###  Question 3
Imaginez que vous deviez configurer l'API d'un agent IA chargé d'auditer et de corriger du code informatique (Python ou C++). Quels réglages de paramètres choisiriez-vous pour garantir la précision technique tout en évitant les inventions de syntaxe ? Justifiez votre choix.